# <font color="#418FDE" size="6.5" uppercase>**Training and Testing**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Run a complete manual training and testing workflow on a small engineering dataset. 
- Recognize overfitting and underfitting from train-test performance gaps and plots. 
- Compare candidate models using consistent metrics and engineering judgment. 


## **1. Training Testing Workflow**

### **1.1. Prepare Engineering Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_01_01.jpg?v=1783662358" width="250">



>* Inspect rows, columns, variables, and units
>* Use engineering judgment to define prediction task

>* Check data errors before splitting
>* Use judgment and encode categories carefully

>* Split data for fair model testing
>* Prevent leakage during data transformations



In [ ]:
#@title Python Code - Prepare Engineering Data

# Prepare small engineering data.
# Inspect quality before splitting.
# Prevent leakage during scaling.

# Import common analysis libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic sample measurements.
rng = np.random.default_rng(7)
rows = 24

# Build a tiny pump dataset.
data = pd.DataFrame({
    "temperature_C": rng.normal(72, 6, rows),
    "vibration_mm_s": rng.normal(4.2, 0.8, rows),
    "pressure_kPa": rng.normal(310, 25, rows),
    "hours_since_service": rng.integers(20, 220, rows),
})

# Add a physically meaningful target.
data["wear_score"] = (
    0.35 * data["temperature_C"]
    + 4.5 * data["vibration_mm_s"]
    + 0.04 * data["hours_since_service"]
)

# Insert realistic data quality problems.
data.loc[3, "temperature_C"] = np.nan
data.loc[8, "pressure_kPa"] = -50

# Count common preparation issues.
missing_count = int(data.isna().sum().sum())
impossible_pressure = int((data["pressure_kPa"] <= 0).sum())

# Repair simple issues using engineering judgment.
data["temperature_C"] = data["temperature_C"].fillna(
    data["temperature_C"].median()
)

# Remove impossible pressure records.
data = data[data["pressure_kPa"] > 0].reset_index(drop=True)

# Separate inputs and target column.
features = ["temperature_C", "vibration_mm_s", "pressure_kPa", "hours_since_service"]
target = "wear_score"

# Shuffle before manual train test split.
shuffled = data.sample(frac=1, random_state=7).reset_index(drop=True)
split_index = int(0.75 * len(shuffled))

# Create held out testing data.
train = shuffled.iloc[:split_index].copy()
test = shuffled.iloc[split_index:].copy()

# Scale using training statistics only.
train_mean = train[features].mean()
train_std = train[features].std(ddof=0).replace(0, 1)

# Apply identical scaling to both portions.
train_scaled = (train[features] - train_mean) / train_std
test_scaled = (test[features] - train_mean) / train_std

# Validate prepared shapes before modeling.
assert len(train) > len(test) > 0
assert train_scaled.shape[1] == len(features)

# Print a compact preparation summary.
print(f"Rows after cleaning: {len(data)}")
print(f"Missing values found: {missing_count}")
print(f"Impossible pressures removed: {impossible_pressure}")
print(f"Train rows: {len(train)}, test rows: {len(test)}")
print("Scaling used training statistics only.")

# Plot train and test target coverage.
plt.figure(figsize=(6, 4))
plt.hist(train[target], alpha=0.7, label="train")
plt.hist(test[target], alpha=0.7, label="test")
plt.xlabel("Pump wear score")
plt.ylabel("Record count")
plt.title("Prepared train test target coverage")
plt.legend()
plt.tight_layout()
plt.show()



### **1.2. Fit Model**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_01_02.jpg?v=1783662361" width="250">



>* Train algorithms using prepared training data only
>* Keep test data unseen for fair evaluation

>* Choose model complexity for task and data
>* Fit candidates consistently to avoid unfair comparisons

>* Check fitted model before test evaluation
>* Ensure learned patterns are physically reasonable



In [ ]:
#@title Python Code - Fit Model

# Fit one simple engineering model.
# Keep test data completely unseen.
# Inspect fitted parameters before testing.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny beam deflection dataset.
rng = np.random.default_rng(7)
loads = np.linspace(100, 900, 18)
lengths = np.linspace(1.0, 2.6, 18)

# Simulate deflection in millimeters.
noise = rng.normal(0, 0.35, loads.size)
deflection = 0.004 * loads + 1.8 * lengths + noise

# Store features and target together.
data = pd.DataFrame({"load_N": loads, "length_m": lengths, "deflection_mm": deflection})
train = data.iloc[:13].copy()
test = data.iloc[13:].copy()

# Build the training design matrix.
X_train = train[["load_N", "length_m"]].to_numpy()
y_train = train["deflection_mm"].to_numpy()

# Validate the training data shape.
assert X_train.shape[0] == y_train.shape[0]
assert X_train.shape[1] == 2

# Add an intercept column manually.
ones = np.ones((X_train.shape[0], 1))
X_design = np.column_stack((ones, X_train))

# Fit linear regression using least squares.
coefficients = np.linalg.lstsq(X_design, y_train, rcond=None)[0]
intercept, load_coef, length_coef = coefficients

# Predict only on training data now.
train_pred = X_design @ coefficients
train_rmse = np.sqrt(np.mean((y_train - train_pred) ** 2))

# Print a compact fitting checkpoint.
print("Manual linear model fitted on training data only.")
print(f"Intercept: {intercept:.3f} mm")
print(f"Load coefficient: {load_coef:.5f} mm per N")
print(f"Length coefficient: {length_coef:.3f} mm per m")
print(f"Training RMSE checkpoint: {train_rmse:.3f} mm")

# Plot fitted training predictions.
plt.figure(figsize=(6, 4))
plt.scatter(y_train, train_pred, color="steelblue", label="training cases")
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], "k--")
plt.xlabel("Measured deflection, mm")
plt.ylabel("Fitted prediction, mm")
plt.title("Fit Check Before Test Evaluation")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Evaluate Test Results**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_01_03.jpg?v=1783662363" width="250">



>* Test on unseen engineering cases
>* Compare predictions using the chosen metric

>* Combine metrics with engineering meaning
>* Inspect large errors for hidden weaknesses

>* Use test results to choose next steps
>* Document metrics, errors, and engineering acceptability



In [ ]:
#@title Python Code - Evaluate Test Results

# Evaluate test results after manual training.
# Small engineering data keeps workflow transparent.
# Metrics and plots support engineering judgment.

import numpy as np
import matplotlib.pyplot as plt

# Create repeatable concrete strength measurements.
rng = np.random.default_rng(7)

# Define water cement ratios for samples.
ratio = np.linspace(0.32, 0.72, 24)

# Simulate compressive strength in megapascals.
strength = 82 - 70 * ratio + rng.normal(0, 2.5, 24)

# Split data manually into train and test.
train_x, test_x = ratio[:16], ratio[16:]
train_y, test_y = strength[:16], strength[16:]

# Validate matching train and test lengths.
assert train_x.size == train_y.size
assert test_x.size == test_y.size

# Fit a straight line using numpy.
slope, intercept = np.polyfit(train_x, train_y, 1)

# Predict both training and test strengths.
train_pred = slope * train_x + intercept
test_pred = slope * test_x + intercept

# Compute mean absolute error values.
train_mae = np.mean(np.abs(train_y - train_pred))
test_mae = np.mean(np.abs(test_y - test_pred))

# Find the largest test prediction error.
test_errors = np.abs(test_y - test_pred)
worst_index = int(np.argmax(test_errors))

# Print compact evaluation results.
print(f"Manual linear model: strength = {slope:.1f}*ratio + {intercept:.1f}")
print(f"Train MAE: {train_mae:.2f} MPa")
print(f"Test MAE: {test_mae:.2f} MPa")
print(f"Performance gap: {test_mae - train_mae:.2f} MPa")
print(f"Worst test error: {test_errors[worst_index]:.2f} MPa")

# Interpret the train test gap simply.
if test_mae > 2.0 * train_mae:
    print("Warning: test error is much larger than training error.")
else:
    print("Train and test errors are reasonably close.")

# Plot actual and predicted test results.
plt.figure(figsize=(6, 4))
plt.scatter(train_x, train_y, label="train actual", color="tab:blue")
plt.scatter(test_x, test_y, label="test actual", color="tab:orange")

# Draw the fitted model line.
all_x = np.linspace(ratio.min(), ratio.max(), 100)
all_y = slope * all_x + intercept
plt.plot(all_x, all_y, label="trained model", color="black")

# Label the engineering plot clearly.
plt.xlabel("Water-cement ratio")
plt.ylabel("Compressive strength (MPa)")
plt.title("Evaluate test results after training")
plt.legend()

# Show the single required plot.
plt.tight_layout()
plt.show()



## **2. Fit Quality Gaps**

### **2.1. Underfitting Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_02_01.jpg?v=1783662352" width="250">



>* Simple models miss important data patterns
>* High train and test errors signal underfitting

>* Plots reveal compressed predictions and patterned errors
>* Underfitting misses important nonlinear structure

>* Poor models may lack key engineering drivers
>* Similar errors can still mean underfitting



In [ ]:
#@title Python Code - Underfitting Signs

# Demonstrate underfitting with simple engineering data.
# Compare training and testing errors carefully.
# Plot predictions and residual patterns clearly.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic random behavior.
rng = np.random.default_rng(7)

# Create small force values in kilonewtons.
force = np.linspace(0, 10, 40)

# Create nonlinear deformation in millimeters.
noise = rng.normal(0, 0.35, force.size)
deformation = 1.5 + 0.35 * force + 0.18 * force**2 + noise

# Split data into training and testing sets.
train_mask = np.arange(force.size) % 4 != 0
test_mask = np.logical_not(train_mask)

# Build a too-simple constant model.
train_mean = deformation[train_mask].mean()
pred_train = np.full(train_mask.sum(), train_mean)
pred_test = np.full(test_mask.sum(), train_mean)

# Define root mean squared error.
def rmse(actual_values, predicted_values):
    squared_errors = (actual_values - predicted_values) ** 2
    return np.sqrt(squared_errors.mean())

# Measure errors on both data splits.
train_error = rmse(deformation[train_mask], pred_train)
test_error = rmse(deformation[test_mask], pred_test)

# Print compact diagnostic results.
print("Manual constant model, no ML library used.")
print(f"Training RMSE: {train_error:.2f} mm")
print(f"Testing RMSE:  {test_error:.2f} mm")
print(f"Train-test gap: {abs(train_error - test_error):.2f} mm")
print("Small gap plus high errors suggests underfitting.")

# Prepare predictions for all observations.
all_predictions = np.full(force.size, train_mean)
residuals = deformation - all_predictions

# Plot actual values and residual pattern.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(force, deformation, label="Actual", color="steelblue")
axes[0].plot(force, all_predictions, label="Constant prediction", color="crimson")
axes[0].set_xlabel("Applied force (kN)")

axes[0].set_ylabel("Deformation (mm)")
axes[0].set_title("Predictions are compressed")
axes[0].legend()
axes[1].scatter(force, residuals, color="darkorange")

axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_xlabel("Applied force (kN)")
axes[1].set_ylabel("Residual (mm)")
axes[1].set_title("Patterned residuals show missed structure")

# Display the single teaching figure.
plt.tight_layout()
plt.show()



### **2.2. Overfitting Signals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_02_02.jpg?v=1783662354" width="250">



>* Excellent training results, poor test results
>* Model memorizes noise instead of generalizing

>* Plots reveal train-test performance divergence
>* Training precision may not generalize

>* Judge train-test gaps with engineering context
>* Simplify or validate before deployment



In [ ]:
#@title Python Code - Overfitting Signals

# Demonstrate overfitting using simple polynomial models.
# Compare training and testing errors visually.
# Use only small synthetic engineering data.

# No extra installations are required.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Make the example deterministic.
rng = np.random.default_rng(7)

# Create small beam test measurements.
x_all = np.linspace(1.0, 10.0, 36)
noise = rng.normal(0.0, 1.8, x_all.size)

# Simulate load capacity in kilonewtons.
y_all = 8.0 + 3.2 * x_all - 0.18 * x_all**2 + noise

# Split alternating points into train and test.
train_mask = np.arange(x_all.size) % 2 == 0
test_mask = np.logical_not(train_mask)

# Store training and testing arrays.
x_train = x_all[train_mask]
y_train = y_all[train_mask]
x_test = x_all[test_mask]
y_test = y_all[test_mask]

# Validate the split before fitting.
assert x_train.size > 10 and x_test.size > 10

# Define a compact RMSE helper.
def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

# Fit a simple stable model.
simple_coef = np.polyfit(x_train, y_train, deg=2)
simple_model = np.poly1d(simple_coef)

# Fit an overly flexible model.
complex_coef = np.polyfit(x_train, y_train, deg=12)
complex_model = np.poly1d(complex_coef)

# Evaluate both models consistently.
simple_train = rmse(y_train, simple_model(x_train))
simple_test = rmse(y_test, simple_model(x_test))

# Evaluate the flexible model consistently.
complex_train = rmse(y_train, complex_model(x_train))
complex_test = rmse(y_test, complex_model(x_test))

# Print only the key comparison lines.
print("Model comparison using RMSE, lower is better.")
print(f"Simple model: train {simple_train:.2f}, test {simple_test:.2f}")
print(f"Flexible model: train {complex_train:.2f}, test {complex_test:.2f}")
print("Overfitting signal: very low train error, much higher test error.")

# Prepare smooth curves for one plot.
x_grid = np.linspace(x_all.min(), x_all.max(), 250)
y_simple = simple_model(x_grid)
y_complex = complex_model(x_grid)

# Plot data and fitted curves.
plt.figure(figsize=(8, 5))
plt.scatter(x_train, y_train, label="training tests", color="tab:blue")
plt.scatter(x_test, y_test, label="new tests", color="tab:orange")

# Add both model curves.
plt.plot(x_grid, y_simple, label="simple quadratic", color="tab:green")
plt.plot(x_grid, y_complex, label="flexible degree 12", color="tab:red")

# Label the engineering context.
plt.xlabel("Specimen thickness, centimeters")
plt.ylabel("Measured load capacity, kN")
plt.title("Overfitting signal: training fit does not generalize")

# Finish the single visible plot.
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **2.3. Test Gap Trends**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_02_03.jpg?v=1783662356" width="250">



>* Track train-test gaps across modeling changes
>* Widening gaps signal possible overfitting

>* Low complexity can hide underfitting
>* Choose strongest test score with acceptable gap

>* Interpret gaps using engineering context
>* Check trends for reliable generalization



In [ ]:
#@title Python Code - Test Gap Trends

# Explore train test gap trends.
# Use polynomial models without libraries.
# Compare complexity using engineering judgment.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatability.
rng = np.random.default_rng(42)

# Create small pump energy dataset.
n_samples = 60
flow_gpm = np.linspace(20, 120, n_samples)

# Simulate nonlinear energy behavior.
noise = rng.normal(0, 18, n_samples)
energy_kw = 35 + 0.9 * flow_gpm + 0.006 * flow_gpm**2 + noise

# Shuffle before making train test split.
indices = rng.permutation(n_samples)
train_idx = indices[:42]
test_idx = indices[42:]

# Validate split sizes before modeling.
assert len(train_idx) > 10 and len(test_idx) > 10

# Build train and test arrays.
x_train = flow_gpm[train_idx]
y_train = energy_kw[train_idx]
x_test = flow_gpm[test_idx]
y_test = energy_kw[test_idx]

# Define root mean squared error.
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# Try increasing polynomial complexity levels.
degrees = list(range(1, 11))
train_errors = []
test_errors = []

# Fit each model silently using numpy.
for degree in degrees:
    coeffs = np.polyfit(x_train, y_train, degree)
    train_pred = np.polyval(coeffs, x_train)
    test_pred = np.polyval(coeffs, x_test)

    train_errors.append(rmse(y_train, train_pred))
    test_errors.append(rmse(y_test, test_pred))

# Find best test performance candidate.
best_index = int(np.argmin(test_errors))
best_degree = degrees[best_index]
best_gap = test_errors[best_index] - train_errors[best_index]

# Print compact comparison summary.
print(f"NumPy version: {np.__version__}")
print("Degree | Train RMSE | Test RMSE | Gap")

# Show only selected complexity levels.
for i in [0, 1, best_index, 9]:
    gap_value = test_errors[i] - train_errors[i]
    print(f"{degrees[i]:>6} | {train_errors[i]:>10.1f} | {test_errors[i]:>9.1f} | {gap_value:>4.1f}")

# Interpret the observed gap trend.
print(f"Best test RMSE occurs near degree {best_degree}.")
print(f"Its test minus train gap is {best_gap:.1f} kW.")

# Plot train and test error trends.
plt.figure(figsize=(7, 4))
plt.plot(degrees, train_errors, marker="o", label="Train RMSE")
plt.plot(degrees, test_errors, marker="s", label="Test RMSE")

# Mark the best test model.
plt.axvline(best_degree, color="gray", linestyle="--", label="Best test degree")
plt.xlabel("Polynomial complexity degree")
plt.ylabel("Prediction error RMSE, kW")
plt.title("Train-test gap trend for pump energy prediction")

# Finish the single teaching plot.
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



## **3. Model Comparison**

### **3.1. Consistent Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_03_01.jpg?v=1783662345" width="250">



>* Use the same metrics for every model
>* Test models under identical conditions

>* Choose metrics that reflect critical mistakes
>* Apply the same metrics to every model

>* Consistent metrics clarify model tradeoffs
>* Support fair, defensible engineering decisions



In [ ]:
#@title Python Code - Consistent Metrics

# Compare models using identical evaluation metrics.
# Use a tiny pump lifetime dataset.
# Keep outputs short for beginners.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic engineering measurements.
hours = np.array([100, 200, 300, 400, 500, 600, 700, 800])
actual_life = np.array([920, 850, 790, 710, 650, 590, 520, 460])

# Store data in a small table.
data = pd.DataFrame({"hours": hours, "life": actual_life})
train = data.iloc[:6].copy()
test = data.iloc[6:].copy()

# Fit two candidate polynomial models.
linear_coef = np.polyfit(train["hours"], train["life"], 1)
quad_coef = np.polyfit(train["hours"], train["life"], 2)

# Predict test values using both models.
linear_pred = np.polyval(linear_coef, test["hours"])
quad_pred = np.polyval(quad_coef, test["hours"])

# Define consistent regression metrics.
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

# Define another consistent metric.
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# Evaluate every model on identical test data.
results = pd.DataFrame({
    "model": ["linear", "quadratic"],
    "MAE_hours": [mae(test["life"], linear_pred), mae(test["life"], quad_pred)],
    "RMSE_hours": [rmse(test["life"], linear_pred), rmse(test["life"], quad_pred)]
})

# Print compact comparison results.
print("Consistent test metrics for pump life prediction:")
print(results.round(1).to_string(index=False))
print("Same split and same metrics make comparison fair.")

# Plot actual data and candidate predictions.
plt.figure(figsize=(7, 4))
plt.scatter(data["hours"], data["life"], label="actual data")
plt.plot(test["hours"], linear_pred, marker="o", label="linear test prediction")
plt.plot(test["hours"], quad_pred, marker="s", label="quadratic test prediction")

# Label the engineering plot clearly.
plt.xlabel("Pump operating hours")
plt.ylabel("Remaining useful life, hours")
plt.title("Model comparison using consistent metrics")
plt.legend()

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **3.2. Engineering Judgment**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_03_02.jpg?v=1783662350" width="250">



>* Use metrics, but do not rely blindly
>* Choose models that behave sensibly

>* Consider which errors matter most
>* Use detailed reviews beyond summary metrics

>* Consider real deployment limits
>* Balance metrics with engineering judgment



In [ ]:
#@title Python Code - Engineering Judgment

# Compare models using engineering judgment.
# Metrics alone cannot choose safely.
# Small data keeps workflow transparent.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic random behavior.
rng = np.random.default_rng(7)

# Create defect sizes in millimeters.
defect_mm = np.linspace(0.2, 5.0, 24)

# Simulate strength with realistic noise.
noise_mpa = rng.normal(0.0, 8.0, defect_mm.size)
strength_mpa = 260.0 - 28.0 * defect_mm + noise_mpa

# Split data into training and testing.
train_mask = np.arange(defect_mm.size) % 4 != 0
test_mask = np.logical_not(train_mask)

# Validate the split sizes.
assert train_mask.sum() > 5 and test_mask.sum() > 3

# Fit simple and complex polynomials.
linear_fit = np.polyfit(defect_mm[train_mask], strength_mpa[train_mask], 1)
complex_fit = np.polyfit(defect_mm[train_mask], strength_mpa[train_mask], 5)

# Predict test strengths consistently.
linear_pred = np.polyval(linear_fit, defect_mm[test_mask])
complex_pred = np.polyval(complex_fit, defect_mm[test_mask])

# Compute mean absolute errors.
linear_mae = np.mean(np.abs(linear_pred - strength_mpa[test_mask]))
complex_mae = np.mean(np.abs(complex_pred - strength_mpa[test_mask]))

# Check physical trend direction.
grid_mm = np.linspace(0.2, 5.0, 100)
linear_grid = np.polyval(linear_fit, grid_mm)
complex_grid = np.polyval(complex_fit, grid_mm)

# Count impossible increasing segments.
linear_bad = np.sum(np.diff(linear_grid) > 0)
complex_bad = np.sum(np.diff(complex_grid) > 0)

# Print compact comparison results.
print(f"Linear MAE: {linear_mae:.1f} MPa, bad trend steps: {linear_bad}")
print(f"Complex MAE: {complex_mae:.1f} MPa, bad trend steps: {complex_bad}")

# Apply a simple engineering decision rule.
if complex_mae + 2.0 < linear_mae and complex_bad == 0:
    choice = "complex polynomial"
else:
    choice = "linear model"

# Explain the selected model briefly.
print(f"Engineering choice: {choice}")
print("Reason: balance test error with physically sensible behavior.")

# Plot data and candidate models.
plt.figure(figsize=(7, 4))
plt.scatter(defect_mm[train_mask], strength_mpa[train_mask], label="train data")
plt.scatter(defect_mm[test_mask], strength_mpa[test_mask], label="test data")
plt.plot(grid_mm, linear_grid, label="linear model")
plt.plot(grid_mm, complex_grid, label="complex model")
plt.xlabel("Defect size (mm)")
plt.ylabel("Strength (MPa)")
plt.title("Model comparison needs engineering judgment")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **3.3. Engineering Judgment**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_03/Lecture_B/image_03_03.jpg?v=1783662347" width="250">



>* Choose models for purpose, not scores alone
>* Prioritize safe, reliable real-world decisions

>* Look beyond average error size
>* Check failures against real engineering risks

>* Balance accuracy with practical deployment constraints
>* Prefer maintainable models that support safe decisions



In [ ]:
#@title Python Code - Engineering Judgment

# Compare models using engineering judgment.
# Metrics alone cannot choose safely.
# Critical operating regions matter most.

import numpy as np
import matplotlib.pyplot as plt

# Create a small pump dataset.
rng = np.random.default_rng(7)
hours = np.linspace(0, 1000, 40)

# Simulate remaining useful life.
noise = rng.normal(0, 18, hours.size)
rul = 1000 - hours + noise

# Split data into train and test.
train_mask = hours <= 700
test_mask = hours > 700

# Fit two candidate polynomial models.
linear_coef = np.polyfit(hours[train_mask], rul[train_mask], 1)
cubic_coef = np.polyfit(hours[train_mask], rul[train_mask], 3)

# Predict on the test region.
linear_pred = np.polyval(linear_coef, hours[test_mask])
cubic_pred = np.polyval(cubic_coef, hours[test_mask])

# Compute consistent test metrics.
true_test = rul[test_mask]
linear_mae = np.mean(np.abs(linear_pred - true_test))
cubic_mae = np.mean(np.abs(cubic_pred - true_test))

# Check conservative behavior near failure.
critical = true_test < 250
linear_under = np.mean(linear_pred[critical] > true_test[critical])
cubic_under = np.mean(cubic_pred[critical] > true_test[critical])

# Print a compact comparison table.
print("Candidate model comparison for pump RUL")
print(f"Linear MAE: {linear_mae:.1f} hours")
print(f"Cubic MAE:  {cubic_mae:.1f} hours")
print(f"Linear risky overprediction rate: {linear_under:.0%}")
print(f"Cubic risky overprediction rate:  {cubic_under:.0%}")

# Apply engineering judgment explicitly.
choice = "linear" if linear_under <= cubic_under else "cubic"
print(f"Engineering choice: {choice} model")
print("Reason: prefer safer near-failure behavior.")

# Plot predictions in the critical region.
plt.figure(figsize=(7, 4))
plt.scatter(hours, rul, label="measured data", color="black")

# Draw both candidate model curves.
grid = np.linspace(0, 1000, 120)
plt.plot(grid, np.polyval(linear_coef, grid), label="linear model")
plt.plot(grid, np.polyval(cubic_coef, grid), label="cubic model")

# Highlight the critical operating region.
plt.axvspan(750, 1000, color="red", alpha=0.12, label="critical region")
plt.xlabel("Pump operating time, hours")
plt.ylabel("Remaining useful life, hours")

# Finish the single teaching plot.
plt.title("Engineering judgment compares metrics and risk")
plt.legend()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Training and Testing**</font>


In this lecture, you learned to:
- Run a complete manual training and testing workflow on a small engineering dataset. 
- Recognize overfitting and underfitting from train-test performance gaps and plots. 
- Compare candidate models using consistent metrics and engineering judgment. 

In the next Module (Module 4), we will go over 'None'